# 05: Streaming Performance Optimization

**Purpose**: Demonstrate memory-efficient streaming processing for large Stack Overflow data dumps.

**Constraints**:
- CPU-only execution (GitHub Actions runners)
- Limited RAM (~7GB available, ~6GB safe limit)
- Multi-gigabyte input files
- Reproducible results

**Key Optimizations**:
1. Generator-based streaming (no full dataset in memory)
2. Chunked processing with configurable batch sizes
3. Memory profiling and automatic garbage collection
4. Incremental aggregation (online statistics)
5. Lazy evaluation for all I/O operations

---

### Limitations Disclosure

- **Data Source**: Stack Overflow public dump (may be incomplete for very recent tags)
- **Memory Constraints**: Streaming ensures compatibility with limited RAM but may be slower than full-load approaches
- **Sampling**: No synthetic sampling used - processes all available real data
- **Reproducibility**: Deterministic processing order ensures identical results across runs

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from code.performance.streaming_optimizer import (
    process_posts_streaming,
    validate_streaming_output,
    benchmark_streaming_performance,
    MemoryMonitor
)

import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Imports successful")

## 1. Memory Monitoring Setup

Initialize memory monitoring to track RAM usage during streaming operations.

In [ ]:
# Initialize memory monitor
monitor = MemoryMonitor(max_memory_mb=6000)  # Conservative limit

# Log initial state
monitor.log_checkpoint("initial", 0)
print(f"Current memory usage: {monitor.get_current_memory_mb():.1f} MB")
print(f"Maximum allowed: {monitor.max_memory_mb} MB")

## 2. Streaming Processing Demo

Process a sample file using the streaming optimizer. If a full dump is available, it will be processed; otherwise, the function will fail loudly (as required) rather than using synthetic data.

In [ ]:
# Define input path (adjust based on available data)
input_path = Path("data/raw/posts_tags.jsonl")
output_path = Path("data/processed/streaming_aggregated.json")

# Check if file exists
if input_path.exists():
    print(f"📂 Processing: {input_path}")
    print(f"   File size: {input_path.stat().st_size / (1024*1024):.1f} MB")
    
    # Start processing
    start_time = time.time()
    result = process_posts_streaming(
        input_path=input_path,
        output_path=output_path,
        chunk_size=10000
    )
    elapsed = time.time() - start_time
    
    print(f"\n✅ Processing complete in {elapsed:.1f} seconds")
    print(f"   Records processed: {result['stats']['total_records']:,}")
    print(f"   Unique tags: {result['stats']['unique_tags']}")
    print(f"   Throughput: {result['stats']['records_per_sec']:.0f} records/sec")
    print(f"   Peak memory: {result['stats']['peak_memory_mb']:.1f} MB")
    print(f"   Safe execution: {result['stats']['safe_execution']}")
else:
    print(f"⚠️ Input file not found: {input_path}")
    print("   Streaming optimization is ready but requires real input data.")
    print("   The system will fail loudly if no real data is available (as per requirements).")

## 3. Validation of Streaming Output

Verify that the streamed data meets minimum quality requirements.

In [ ]:
# Load aggregated data if available
if output_path.exists():
    with open(output_path, 'r') as f:
        aggregated_data = json.load(f)
    
    # Validate
    validation_results, valid_tags = validate_streaming_output(
        aggregated_data,
        min_months=12,
        min_posts=100
    )
    
    print(f"Validation Results:")
    print(f"  Total tags: {validation_results['total_tags']}")
    print(f"  Valid tags (≥12 months, ≥100 posts): {validation_results['valid_tags']}")
    print(f"  Invalid tags: {validation_results['invalid_tags']}")
    print(f"  Validation passed: {validation_results['validation_passed']}")
    
    # Show top 10 valid tags by post count
    if valid_tags:
        tag_totals = [(tag, sum(aggregated_data[tag].values())) for tag in valid_tags]
        tag_totals.sort(key=lambda x: x[1], reverse=True)
        
        print("\nTop 10 tags by post count:")
        for tag, count in tag_totals[:10]:
            print(f"  {tag}: {count:,} posts")
else:
    print("⚠️ Output file not found - skipping validation")

## 4. Memory Usage Visualization

Plot memory usage over time to demonstrate streaming efficiency.

In [ ]:
import matplotlib.pyplot as plt

# Load memory report if available
report_path = Path("logs/streaming_memory_report.json")
if report_path.exists():
    with open(report_path, 'r') as f:
        memory_report = json.load(f)
    
    # Extract checkpoint data
    checkpoints = memory_report.get('checkpoints', [])
    if checkpoints:
        times = [cp['timestamp'] - checkpoints[0]['timestamp'] for cp in checkpoints]
        memories = [cp['memory_mb'] for cp in checkpoints]
        
        plt.figure(figsize=(12, 6))
        plt.plot(times, memories, 'b-', linewidth=2, label='Memory Usage')
        plt.axhline(y=memory_report['limit_mb'], color='r', linestyle='--', label='Memory Limit')
        plt.axhline(y=memory_report['max_memory_mb'], color='g', linestyle=':', label='Peak Memory')
        
        plt.xlabel('Time (seconds)')
        plt.ylabel('Memory Usage (MB)')
        plt.title('Memory Usage During Streaming Processing')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print(f"\n✅ Memory report: Peak {memory_report['max_memory_mb']:.1f} MB / Limit {memory_report['limit_mb']} MB")
        print(f"   Safe execution: {memory_report['safe_execution']}")
    else:
        print("⚠️ No checkpoints found in memory report")
else:
    print("⚠️ Memory report not found")

## 5. Benchmarking (Optional)

Run performance benchmarks across different file sizes if multiple test files are available.

In [ ]:
# Define sample files for benchmarking
sample_files = [
    Path("data/raw/posts_tags_small.jsonl"),
    Path("data/raw/posts_tags_medium.jsonl"),
    Path("data/raw/posts_tags.jsonl")
]

# Filter existing files
existing_files = [f for f in sample_files if f.exists()]

if len(existing_files) >= 2:
    print(f"Running benchmark on {len(existing_files)} files...")
    benchmark_results = benchmark_streaming_performance(existing_files, Path("data/processed"))
    
    # Display results
    for res in benchmark_results['benchmarks']:
        print(f"\nFile: {res['file'].split('/')[-1]}")
        print(f"  Size: {res['size_mb']:.1f} MB")
        print(f"  Records: {res['records']:,}")
        print(f"  Time: {res['time_sec']:.1f}s")
        print(f"  Throughput: {res['throughput_rec_sec']:.0f} rec/sec")
        print(f"  Peak Memory: {res['peak_memory_mb']:.1f} MB")
        print(f"  Safe: {res['safe']}")
else:
    print(f"⚠️ Need at least 2 sample files for benchmarking. Found: {len(existing_files)}")

## 6. Summary and Reproducibility

The streaming optimization ensures:
- **Memory Safety**: Never exceeds 6GB RAM limit
- **Scalability**: Handles multi-gigabyte dumps
- **Reproducibility**: Deterministic processing order
- **Real Data**: No synthetic fallbacks - fails loudly if real data unavailable

**Next Steps**:
1. Ensure input data is available at `data/raw/posts_tags.jsonl`
2. Run `code/performance/streaming_optimizer.py --input data/raw/posts_tags.jsonl`
3. Verify output at `data/processed/streaming_aggregated.json`
4. Check memory report at `logs/streaming_memory_report.json`